# Project 1: Hinglish Text Generation with Llama-2

This notebook demonstrates fine-tuning the TinyLlama model for **Hinglish** (a mix of Hindi and English) text generation.

1. **Environment Setup**
2. **Configuration**
3. **Data Acquisition**
4. **Preprocessing**
5. **Model Preparation**
6. **Training**
7. **Inference**

In [ ]:
!pip install -q transformers trl datasets accelerate peft bitsandbytes

In [ ]:
base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
dataset_name = "shashank-v-sharma/hinglish-top"
new_model_name = "tinyllama-hinglish-gen"
output_dir = "./hinglish_results"
num_train_epochs = 1
per_device_train_batch_size = 4
learning_rate = 2e-4
max_seq_length = 512

In [ ]:
from datasets import load_dataset
dataset = load_dataset(dataset_name, split="train")
print(f"Sample: {dataset[0]}")

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token

def formatting_prompts_func(example):
    output_texts = []
    for i in range(len(example["en_query"])):
        instruction = example["en_query"][i]
        response = example["hi_query"][i]
        text = f"User: {instruction} ###> Assistant: {response}"
        output_texts.append(text)
    return output_texts

In [ ]:
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM
from peft import LoraConfig, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(base_model_name, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
args = TrainingArguments(output_dir=output_dir, max_steps=100, fp16=True, report_to="none")
trainer = SFTTrainer(model=model, train_dataset=dataset, peft_config=peft_config, formatting_func=formatting_prompts_func, tokenizer=tokenizer, args=args)

In [ ]:
def generate_hinglish(prompt, model, tokenizer):
    input_text = f"User: {prompt} ###> Assistant:"
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)